In [1]:
import sys
sys.path.append('..')

import pandas as pd
import requests
import time
from src.config import (
    RAW_OPENAQ_DIR,
    OPENAQ_API_KEY,
    OPENAQ_BASE_URL,
    JAKARTA_MONITOR_NAMES,
    DATE_FROM,
    DATE_TO,
)
from src.data.fetch_openaq import fetch_location_history

HEADERS = {"X-API-Key": OPENAQ_API_KEY}
print(f"API Key loaded: {'YES' if OPENAQ_API_KEY else 'NO'}")
print(f"Date range: {DATE_FROM} to {DATE_TO}")
print(f"Target stations: {JAKARTA_MONITOR_NAMES}")

API Key loaded: YES
Date range: 2019-01-01 to 2023-12-31
Target stations: ['Jakarta Central', 'Jakarta South', 'West Jakarta Mayor Office', 'East Jakarta Mayor Office', 'Rusunawa Marunda']


In [2]:
def get_jakarta_monitor_ids() -> pd.DataFrame:
    """Fetch location IDs for Jakarta reference monitors via v3 API.
    Uses countries_id=1 (Indonesia) + name matching — the only
    reliable filter in OpenAQ v3 (city/country text params are ignored).
    """
    all_locs = []
    page = 1
    while True:
        r = requests.get(
            f"{OPENAQ_BASE_URL}/locations",
            headers=HEADERS,
            params={"countries_id": 1, "limit": 1000, "page": page},
            timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        results = data.get("results", [])
        if not results:
            break
        all_locs.extend(results)
        if len(results) < 1000:
            break
        page += 1
        time.sleep(0.5)

    print(f"Total Indonesia locations fetched: {len(all_locs)}")

    rows = []
    for loc in all_locs:
        if loc.get("name") in JAKARTA_MONITOR_NAMES:
            sensors = [s["parameter"]["name"] for s in loc.get("sensors", [])]
            dt_first = (loc.get("datetimeFirst") or {}).get("utc")
            dt_last = (loc.get("datetimeLast") or {}).get("utc")
            rows.append({
                "location_id": loc["id"],
                "name": loc["name"],
                "is_monitor": loc.get("isMonitor", False),
                "parameters": sensors,
                "first_updated": dt_first,
                "last_updated": dt_last,
            })

    return pd.DataFrame(rows)


monitor_df = get_jakarta_monitor_ids()
print(f"Found {len(monitor_df)} target monitors:")
monitor_df

Total Indonesia locations fetched: 39
Found 5 target monitors:


,location_id,name,is_monitor,parameters,first_updated,last_updated
0,8320,Jakarta South,True,[pm25],2022-10-03T20:00:00Z,2025-11-04T21:00:00Z
1,8637,Jakarta Central,True,"[o3, pm25]",2016-11-09T18:00:00Z,2025-04-09T14:00:00Z
2,6051931,West Jakarta Mayor Office,True,"[pm25, relativehumidity, temperature, wind_dir...",2023-09-30T17:00:00Z,2025-06-30T16:00:00Z
3,6051932,East Jakarta Mayor Office,True,"[bc_370, bc_880, pm25, relativehumidity, tempe...",2023-09-30T17:00:00Z,2025-06-30T16:00:00Z
4,6051933,Rusunawa Marunda,True,"[pm25, relativehumidity, temperature, wind_dir...",2023-09-30T17:00:00Z,2025-06-30T16:00:00Z


In [3]:
all_dfs = []

for _, row in monitor_df.iterrows():
    loc_id = int(row["location_id"])
    name = row["name"]
    print(f"\n{'='*50}")
    print(f"Fetching: {name} (id={loc_id})")
    print(f"{'='*50}")

    df = fetch_location_history(
        location_id=loc_id,
        city_key="jakarta",
        date_from=DATE_FROM,
        date_to=DATE_TO,
    )

    if not df.empty:
        df["station_name"] = name
        df["location_id"] = loc_id
        all_dfs.append(df)
        print(f"  Records fetched: {len(df):,}")
    else:
        print(f"  No data returned")

    time.sleep(1)

print(f"\n{'='*50}")
print(f"DONE. Total stations fetched: {len(all_dfs)}")


Fetching: Jakarta South (id=8320)
  [CACHE] Loading from cache: jakarta_loc8320_2019-01-01_2023-12-31.parquet


  Records fetched: 235



Fetching: Jakarta Central (id=8637)


  pm25 (sensor_id=25072) page 

1...

...

  pm25 (sensor_id=14313805) page 

1...

 done (591 daily records)


  Records fetched: 91



DONE. Total stations fetched: 5


In [4]:
if all_dfs:
    raw_df = pd.concat(all_dfs, ignore_index=True)

    output_path = RAW_OPENAQ_DIR / "jakarta_raw_combined.parquet"
    raw_df.to_parquet(output_path)

    print(f"Saved to: {output_path}")
    print(f"Total records: {len(raw_df):,}")
    print(f"\nColumns:\n{raw_df.columns.tolist()}")

    # Date range — v3 json_normalize produces 'period.datetimeFrom.utc'
    date_col = next(
        (c for c in raw_df.columns if "datetimeFrom" in c and c.endswith(".utc")),
        None,
    )
    if date_col:
        print(f"\nDate range ({date_col}):")
        print(raw_df[date_col].agg(["min", "max"]))
    else:
        print("\n[WARNING] Could not detect date column — check columns list above")

    # Parameter breakdown — v3 uses 'parameter.name'
    param_col = next(
        (c for c in raw_df.columns if c in ("parameter.name", "parameter")),
        None,
    )
    if param_col:
        print(f"\nParameter breakdown ({param_col}):")
        print(raw_df[param_col].value_counts())
else:
    print("No data fetched — check API key and station IDs")

Saved to: C:\AQSEL\me\KARIR\Portofolio Setelah Wisuda\EDA Project Indonesia Urban Air Quality Analysis\data\raw\openaq\jakarta_raw_combined.parquet
Total records: 2,219

Columns:
['value', 'coordinates', 'parameter_name', 'flagInfo.hasFlags', 'parameter.id', 'parameter.name', 'parameter.units', 'parameter.displayName', 'period.label', 'period.interval', 'period.datetimeFrom.utc', 'period.datetimeFrom.local', 'period.datetimeTo.utc', 'period.datetimeTo.local', 'summary.min', 'summary.q02', 'summary.q25', 'summary.median', 'summary.q75', 'summary.q98', 'summary.max', 'summary.avg', 'summary.sd', 'coverage.expectedCount', 'coverage.expectedInterval', 'coverage.observedCount', 'coverage.observedInterval', 'coverage.percentComplete', 'coverage.percentCoverage', 'coverage.datetimeFrom.utc', 'coverage.datetimeFrom.local', 'coverage.datetimeTo.utc', 'coverage.datetimeTo.local', 'station_name', 'location_id']

Date range (period.datetimeFrom.utc):
min    2019-01-01T17:00:00Z
max    2023-12-31T1